In [6]:
# ============================================================
# 导入库
# ============================================================
import sys
# 把项目根目录加入 Python 路径，确保在 VSCode 中也能找到 util、model 等模块
sys.path.insert(0, '/home/simplexity/cyt/pinnsformer-main')

import os
# 设置 PyTorch 显存分配策略：将大块显存拆分为 128MB 的块，减少碎片
# 1080Ti 等小显存 GPU 配合此设置可缓解 CUDA OOM
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

import time                          # 训练计时
import numpy as np                   # 数值计算、数组操作
import torch                         # 深度学习框架
import torch.nn as nn                # 神经网络模块
import matplotlib.pyplot as plt      # 绘图
import random                        # Python 随机数（种子设置用）
from torch.optim import LBFGS, Adam  # LBFGS: 准牛顿法; Adam: 自适应梯度（备选）
from tqdm import tqdm                # 进度条
import scipy.io                      # 读取 .mat 格式数据

from util import *                   # 导入项目工具函数：get_data, make_time_sequence, get_clones, get_n_params

In [7]:
# ============================================================
# 随机种子 & 设备
# ============================================================
seed = 0
np.random.seed(seed)          # NumPy 随机数种子
random.seed(seed)             # Python 标准库随机数种子
torch.manual_seed(seed)       # PyTorch CPU 随机数种子
torch.cuda.manual_seed(seed)  # PyTorch GPU 随机数种子（保证每次运行结果一致）

device = 'cuda:0'             # 使用第 0 号 GPU；若无 GPU 则改为 'cpu'

# ============================================================
# 可调超参数 — 只需改这里，后续 cell 自动引用，无需手动查找硬编码
# ============================================================

# --- 数据参数 ---
# 训练点数：从全部 N*T ≈ 1,000,000 个时空点中随机采样多少点用于训练
# 点数越大 → 精度越高，但显存消耗也线性增长
# 12GB 显存建议 ≤800，24GB+ 可恢复原始值 2500
N_TRAIN       = 800

# 伪时间序列长度：每个空间点 (x,y) 扩展为多少个时间步
# 序列越长 → 时间上下文越丰富，但计算量 O(NUM_SEQ²) 增长（attention 是二次复杂度）
NUM_SEQ       = 5

# 伪时间步长：相邻时间步之间的间隔 Δt
# 1D PDE（Reaction/Convection/Wave）用 1e-4，Navier-Stokes 时间尺度大，用 1e-2
SEQ_STEP      = 1e-2

# --- 模型结构参数 ---
D_OUT         = 2           # 输出维度：流函数 ψ 和压力 p（固定为 2）
D_MODEL       = 32          # Transformer 嵌入维度（所有 attention 和 FFN 的内部维度）
D_HIDDEN      = 512         # 输出头隐层维度（最后几层 MLP 的宽度）
D_FF          = 256         # FeedForward 内部隐层维度（Transformer 中 FFN 的隐藏层宽度）
N_LAYERS      = 1           # Encoder 和 Decoder 各自堆叠的层数
N_HEADS       = 2           # 多头注意力头数（必须能整除 D_MODEL，32/2=16 每头）

# --- 训练参数 ---
N_EPOCHS      = 1000        # 最大迭代次数（LBFGS 通常 500~2000 轮收敛）
OPTIMIZER     = 'LBFGS'     # 优化器选择：'LBFGS' 收敛快但显存大 / 'Adam' 省显存但慢
LINE_SEARCH   = 'strong_wolfe'  # LBFGS 线搜索策略：'strong_wolfe' 更稳定 / None 省显存

# --- PDE 参数 ---
NU            = 0.01        # Navier-Stokes 方程中的运动粘度 ν（雷诺数 Re ∝ 1/ν）

# --- 评估参数 ---
TEST_SNAP     = 100         # 测试时使用的时间快照索引（数据共 T=200 帧，取第 100 帧）

In [9]:
# ============================================================
# 加载 CFD 仿真数据（圆柱绕流）
# ============================================================
# 数据来源：Nektar++ 谱元法求解器生成的圆柱绕流 DNS 数据
# 包含 5000 个空间点 × 200 个时间步的完整流场
data = scipy.io.loadmat('./cylinder_nektar_wake.mat')

# ============================================================
# 查看数据集所有内容
# ============================================================
print('=' * 55)
print('数据集变量一览:')
print('=' * 55)
for key in data.keys():
    if not key.startswith('__'):
        arr = data[key]
        print(f"  '{key}': {arr.shape}  ({arr.dtype})")

print()

# 逐个变量打印一组样本数据
for key in ['X_star', 't', 'U_star', 'p_star']:
    arr = data[key]
    print('=' * 55)
    print(f"{key}: 形状 {arr.shape}  |  类型 {arr.dtype}")
    print('=' * 55)

    if key == 'X_star':
        # (5000, 2) 空间坐标
        print(f'  第0个点:  x={arr[0,0]:.4f},  y={arr[0,1]:.4f}')
        print(f'  第1个点:  x={arr[1,0]:.4f},  y={arr[1,1]:.4f}')
        print(f'  ...')
        print(f'  第4999个点: x={arr[-1,0]:.4f}, y={arr[-1,1]:.4f}')
        print(f'  数值范围:   x∈[{arr[:,0].min():.4f}, {arr[:,0].max():.4f}]')
        print(f'              y∈[{arr[:,1].min():.4f}, {arr[:,1].max():.4f}]')

    elif key == 't':
        # (200, 1) 时间坐标
        print(f'  t[0]  = {arr[0,0]:.2f}')
        print(f'  t[1]  = {arr[1,0]:.2f}')
        print(f'  t[2]  = {arr[2,0]:.2f}')
        print(f'  ...')
        print(f'  t[199] = {arr[-1,0]:.2f}')
        print(f'  Δt = {arr[1,0]-arr[0,0]:.2f}')

    elif key == 'U_star':
        # (5000, 2, 200) 速度场
        print(f'  第0个空间点, 第0时间步: u={arr[0,0,0]:.6f}, v={arr[0,1,0]:.6f}')
        print(f'  第0个空间点, 第1时间步: u={arr[0,0,1]:.6f}, v={arr[0,1,1]:.6f}')
        print(f'  第1个空间点, 第0时间步: u={arr[1,0,0]:.6f}, v={arr[1,1,0]:.6f}')
        print(f'  ...')
        print(f'  数值范围: u∈[{arr[:,0,:].min():.6f}, {arr[:,0,:].max():.6f}]')
        print(f'            v∈[{arr[:,1,:].min():.6f}, {arr[:,1,:].max():.6f}]')

    elif key == 'p_star':
        # (5000, 200) 压力场
        print(f'  第0个空间点, 第0时间步: p={arr[0,0]:.6f}')
        print(f'  第0个空间点, 第1时间步: p={arr[0,1]:.6f}')
        print(f'  第1个空间点, 第0时间步: p={arr[1,0]:.6f}')
        print(f'  ...')
        print(f'  数值范围: p∈[{arr.min():.6f}, {arr.max():.6f}]')

    print()

数据集变量一览:
  'X_star': (5000, 2)  (float64)
  't': (200, 1)  (float64)
  'U_star': (5000, 2, 200)  (float64)
  'p_star': (5000, 200)  (float64)

X_star: 形状 (5000, 2)  |  类型 float64
  第0个点:  x=1.0000,  y=-2.0000
  第1个点:  x=1.0707,  y=-2.0000
  ...
  第4999个点: x=8.0000, y=2.0000
  数值范围:   x∈[1.0000, 8.0000]
              y∈[-2.0000, 2.0000]

t: 形状 (200, 1)  |  类型 float64
  t[0]  = 0.00
  t[1]  = 0.10
  t[2]  = 0.20
  ...
  t[199] = 19.90
  Δt = 0.10

U_star: 形状 (5000, 2, 200)  |  类型 float64
  第0个空间点, 第0时间步: u=1.114192, v=-0.004096
  第0个空间点, 第1时间步: u=1.117557, v=0.001092
  第1个空间点, 第0时间步: u=1.111027, v=0.000393
  ...
  数值范围: u∈[-0.240268, 1.322556]
            v∈[-0.624111, 0.623436]

p_star: 形状 (5000, 200)  |  类型 float64
  第0个空间点, 第0时间步: p=-0.108155
  第0个空间点, 第1时间步: p=-0.111155
  第1个空间点, 第0时间步: p=-0.105604
  ...
  数值范围: p∈[-0.536476, 0.072474]



In [ ]:
# ============================================================
# 数据预处理：提取、展平、采样、构造伪时间序列
# ============================================================

# --- 从 .mat 文件中提取各个数组 ---
U_star = data['U_star']   # 速度场，形状 (N, 2, T) = (5000 空间点, u/v 两分量, 200 时间步)
P_star = data['p_star']   # 压力场，形状 (N, T)    = (5000 空间点, 200 时间步)
t_star = data['t']        # 时间坐标，形状 (T, 1)   = (200 时间步, 1)
X_star = data['X_star']   # 空间坐标，形状 (N, 2)   = (5000 空间点, x/y 两坐标)
print("U_star 速度场形状:", U_star.shape)
print("U_star", U_star[0][0].shape)
print(f'数据集大小: 空间点数 N={X_star.shape[0]}, 时间步数 T={t_star.shape[0]}')
# print("U_star", U_star[0][1])

N = X_star.shape[0]       # 空间点数 = 5000
T = t_star.shape[0]       # 时间步数 = 200
print(f'数据集大小: 空间点数 N={N}, 时间步数 T={T}')

# --- 将数据展平为 (N*T, 1) 的列向量 ---
# np.tile 沿指定维度复制数组，构造完整的时空网格
XX = np.tile(X_star[:,0:1], (1,T)) # x 坐标复制 T 列，形状 (N, T)
YY = np.tile(X_star[:,1:2], (1,T)) # y 坐标复制 T 列，形状 (N, T)
TT = np.tile(t_star, (1,N)).T      # t 坐标复制 N 行再转置，形状 (N, T)
print("X_star[:,0:1]", X_star[:,0:1])
print("X_star[:,1:2]", X_star[:,1:2])


UU = U_star[:,0,:]  # u 速度分量，形状 (N, T)
VV = U_star[:,1,:]  # v 速度分量，形状 (N, T)
PP = P_star          # 压力，形状 (N, T)
print(f'XX 形状: {XX.shape}, YY 形状: {YY.shape}, TT 形状: {TT.shape}')
print("UU",UU)
print("TT",TT)

# flatten() 按行展开为一维，[:,None] 添加轴变成列向量
x = XX.flatten()[:,None]  # (N*T, 1) = (1,000,000, 1)
y = YY.flatten()[:,None]
t = TT.flatten()[:,None]
u = UU.flatten()[:,None]
v = VV.flatten()[:,None]
p = PP.flatten()[:,None]
print(f'展平后 x 形状: {x.shape}, y 形状: {y.shape}, t 形状: {t.shape}')
print("x:",x)
print("y:",y)
print("t:",t)
print("u:",u)
print("v:",v)
print("p:",p)

# --- 随机采样训练点 ---
# 从 ~1M 个点中不放回随机采样 N_TRAIN 个，大幅减少计算量
idx = np.random.choice(N*T, N_TRAIN, replace=False)
x_train = x[idx,:]  # (N_TRAIN, 1)
y_train = y[idx,:]
t_train = t[idx,:]
u_train = u[idx,:]  # 真实 u 速度，用于数据约束
v_train = v[idx,:]  # 真实 v 速度，用于数据约束

# --- 构造伪时间序列（PINNsformer 的核心预处理步骤）---
# 将每个采样点复制 NUM_SEQ 份，沿最后一维堆叠
# x_train: (N_TRAIN, 1) → (N_TRAIN, NUM_SEQ, 1)，x 坐标在序列维度上不变
x_train = np.expand_dims(np.tile(x_train[:], (NUM_SEQ)) ,-1)
y_train = np.expand_dims(np.tile(y_train[:], (NUM_SEQ)) ,-1)
# make_time_sequence: 把 t 扩展为 [t, t+Δt, t+2Δt, t+3Δt, t+4Δt]
# 输入 (N_TRAIN, 1) → 输出 (N_TRAIN, NUM_SEQ, 1)
t_train = make_time_sequence(t_train, num_step=NUM_SEQ, step=SEQ_STEP)

# --- 转为 PyTorch 张量并上传 GPU ---
# requires_grad=True 是必须的：训练时需要对这些输入求偏导数（PDE residual）
x_train = torch.tensor(x_train, dtype=torch.float32, requires_grad=True).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32, requires_grad=True).to(device)
t_train = torch.tensor(t_train, dtype=torch.float32, requires_grad=True).to(device)
# u_train, v_train 不需要求导（它们是标签值，不是网络输入）
u_train = torch.tensor(u_train, dtype=torch.float32, requires_grad=True).to(device)
v_train = torch.tensor(v_train, dtype=torch.float32, requires_grad=True).to(device)

print(f'训练点数: {N_TRAIN} / 可用总数: {N*T}')

U_star 速度场形状: (5000, 2, 200)
U_star (200,)
数据集大小: 空间点数 N=5000, 时间步数 T=200
数据集大小: 空间点数 N=5000, 时间步数 T=200
X_star[:,0:1] [[1.        ]
 [1.07070707]
 [1.14141414]
 ...
 [7.85858586]
 [7.92929293]
 [8.        ]]
X_star[:,1:2] [[-2.]
 [-2.]
 [-2.]
 ...
 [ 2.]
 [ 2.]
 [ 2.]]
XX 形状: (5000, 200), YY 形状: (5000, 200), TT 形状: (5000, 200)
UU [[1.11419216 1.11755721 1.11956933 ... 1.16105653 1.16209316 1.16287446]
 [1.11102707 1.11424311 1.11623549 ... 1.16119046 1.16250501 1.16355997]
 [1.10748452 1.11049848 1.11244362 ... 1.16080365 1.16241577 1.16375737]
 ...
 [1.04183122 1.07052197 1.08570786 ... 1.03561691 1.01482352 0.99560744]
 [1.03144583 1.05940483 1.07424671 ... 1.05086925 1.02921296 1.00907934]
 [1.02128975 1.04848021 1.06294889 ... 1.06667284 1.044304   1.02333582]]
TT [[ 0.   0.1  0.2 ... 19.7 19.8 19.9]
 [ 0.   0.1  0.2 ... 19.7 19.8 19.9]
 [ 0.   0.1  0.2 ... 19.7 19.8 19.9]
 ...
 [ 0.   0.1  0.2 ... 19.7 19.8 19.9]
 [ 0.   0.1  0.2 ... 19.7 19.8 19.9]
 [ 0.   0.1  0.2 ... 19.7 19.8

: 

In [ ]:
# ============================================================
# PINNsformer 模型架构定义
# ============================================================
# 架构总览（自底向上）：
#   WaveAct → FeedForward → EncoderLayer → DecoderLayer → Encoder → Decoder → PINNsformer
#
# 数据流向：
#   [x,y,t] → linear_emb → Encoder(自注意力) → Decoder(交叉注意力) → linear_out → [ψ,p]
#
# 与标准 Transformer 的两个关键区别：
#   1. 用 WaveAct (可学习 sin/cos) 替代 LayerNorm + ReLU
#   2. 激活在注意力之前，而非之后


# ============================================================
# WaveAct: 小波激活函数
# ============================================================
# 论文核心创新之一。替代传统 Tanh/ReLU。
# 形式: f(x) = w1 * sin(x) + w2 * cos(x)
# w1, w2 是可学习参数（初始化为 1），训练中自动调整振幅比。
# 优势：sin/cos 天然具有高频振荡能力，适合捕捉 PDE 的高频解（如湍流、激波）
class WaveAct(nn.Module):
    def __init__(self):
        super(WaveAct, self).__init__()
        # nn.Parameter 将张量注册为可学习参数，optimizer 会自动追踪和更新
        self.w1 = nn.Parameter(torch.ones(1), requires_grad=True)
        self.w2 = nn.Parameter(torch.ones(1), requires_grad=True)

    def forward(self, x):
        return self.w1 * torch.sin(x) + self.w2 * torch.cos(x)


# ============================================================
# FeedForward: Transformer 中的前馈网络
# ============================================================
# 两层 MLP + WaveAct 激活，中间维度膨胀再收缩
# d_model → d_ff → d_ff → d_model（先升维再降维，增强表达能力）
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=256):
        super(FeedForward, self).__init__()
        self.linear = nn.Sequential(*[
            nn.Linear(d_model, d_ff),   # 升维：32 → 256
            WaveAct(),
            nn.Linear(d_ff, d_ff),      # 保持：256 → 256
            WaveAct(),
            nn.Linear(d_ff, d_model)    # 降维：256 → 32（恢复残差连接维度）
        ])

    def forward(self, x):
        return self.linear(x)


# ============================================================
# EncoderLayer: 编码器单层（自注意力）
# ============================================================
# Pre-norm 风格的 Transformer 层，但用 WaveAct 替代 LayerNorm
# 流程: WaveAct → Self-Attention(残差) → WaveAct → FFN(残差)
class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff=256):
        super(EncoderLayer, self).__init__()
        # batch_first=True: 输入形状为 (batch, seq, feature) 而非 (seq, batch, feature)
        # MultiheadAttention 内部将 d_model 拆分为 heads 份，每份 d_k = d_model/heads
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=heads, batch_first=True)
        self.ff = FeedForward(d_model, d_ff)
        self.act1 = WaveAct()  # 注意力前的激活
        self.act2 = WaveAct()  # FFN 前的激活

    def forward(self, x):
        # ① 先 WaveAct 激活，再做自注意力
        #    Q=K=V=x2（同一个序列自己注意自己，学习序列内部依赖）
        #    attn 返回 (output, attention_weights)，取 [0] 只要输出
        x2 = self.act1(x)
        x = x + self.attn(x2, x2, x2)[0]   # 残差连接：x = x + Attention(WaveAct(x))

        # ② 再 WaveAct 激活，过前馈网络
        x2 = self.act2(x)
        x = x + self.ff(x2)                 # 残差连接：x = x + FFN(WaveAct(x))
        return x


# ============================================================
# DecoderLayer: 解码器单层（交叉注意力）
# ============================================================
# 与 EncoderLayer 的唯一区别：注意力机制从自注意力变为交叉注意力
# 流程: WaveAct → Cross-Attention(Q=x, K=V=encoder_out, 残差) → WaveAct → FFN(残差)
class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff=256):
        super(DecoderLayer, self).__init__()
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=heads, batch_first=True)
        self.ff = FeedForward(d_model, d_ff)
        self.act1 = WaveAct()
        self.act2 = WaveAct()

    def forward(self, x, e_outputs):
        # 关键区别：attn(query=x2, key=e_outputs, value=e_outputs)
        # Q 来自 Decoder 自己的输入（含伪时间序列信息）
        # K,V 来自 Encoder 的输出（含全局时空特征）
        # 让 Decoder 从 Encoder 提取的全局特征中查询相关信息
        x2 = self.act1(x)
        x = x + self.attn(x2, e_outputs, e_outputs)[0]  # 交叉注意力 + 残差

        x2 = self.act2(x)
        x = x + self.ff(x2)                              # FFN + 残差
        return x


# ============================================================
# Encoder: 编码器（堆叠 N 层 EncoderLayer）
# ============================================================
# get_clones 深拷贝 N 份 EncoderLayer，参数独立
class Encoder(nn.Module):
    def __init__(self, d_model, N, heads, d_ff=256):
        super(Encoder, self).__init__()
        self.N = N
        # get_clones: 用 copy.deepcopy 复制 N 份相同结构但参数独立 EncoderLayer
        self.layers = get_clones(EncoderLayer(d_model, heads, d_ff), N)
        self.act = WaveAct()  # 最终输出前再激活一次

    def forward(self, x):
        for i in range(self.N):
            x = self.layers[i](x)
        return self.act(x)


# ============================================================
# Decoder: 解码器（堆叠 N 层 DecoderLayer）
# ============================================================
class Decoder(nn.Module):
    def __init__(self, d_model, N, heads, d_ff=256):
        super(Decoder, self).__init__()
        self.N = N
        self.layers = get_clones(DecoderLayer(d_model, heads, d_ff), N)
        self.act = WaveAct()

    def forward(self, x, e_outputs):
        # 每一层 DecoderLayer 都接收 Encoder 的最终输出 e_outputs 作为 K,V
        for i in range(self.N):
            x = self.layers[i](x, e_outputs)
        return self.act(x)


# ============================================================
# PINNsformer: 顶层模型
# ============================================================
# 组装完整的数据流：
#   输入 [x,y,t] (N,5,3)
#     → linear_emb:   Linear(3, d_model)        维度映射 3→32
#     → encoder:      自注意力提取时空特征
#     → decoder:      交叉注意力融合全局+局部信息
#     → linear_out:   3层 MLP (WaveAct 激活)     维度映射 32→2
#   输出 [ψ, p] (N,5,2)
class PINNsformer(nn.Module):
    def __init__(self, d_out, d_model, d_hidden, d_ff, N, heads):
        super(PINNsformer, self).__init__()

        # 输入嵌入层：将 3 维物理坐标 (x, y, t) 映射到 d_model 维的嵌入空间
        self.linear_emb = nn.Linear(3, d_model)

        # Encoder: 自注意力编码器
        self.encoder = Encoder(d_model, N, heads, d_ff)

        # Decoder: 交叉注意力解码器
        self.decoder = Decoder(d_model, N, heads, d_ff)

        # 输出头：3 层 MLP，将 d_model 维特征映射到 d_out 维输出
        self.linear_out = nn.Sequential(*[
            nn.Linear(d_model, d_hidden),   # 32 → 512
            WaveAct(),
            nn.Linear(d_hidden, d_hidden),  # 512 → 512
            WaveAct(),
            nn.Linear(d_hidden, d_out)      # 512 → 2 (ψ, p)
        ])

    def forward(self, x, y, t):
        # 第 1 步：拼接三个输入通道 → (N, NUM_SEQ, 3)
        src = torch.cat((x, y, t), dim=-1)

        # 第 2 步：线性嵌入 → (N, NUM_SEQ, d_model=32)
        src = self.linear_emb(src)

        # 第 3 步：Encoder 自注意力提取特征 → (N, NUM_SEQ, 32)
        e_outputs = self.encoder(src)

        # 第 4 步：Decoder 交叉注意力融合信息 → (N, NUM_SEQ, 32)
        d_output = self.decoder(src, e_outputs)

        # 第 5 步：输出头映射到物理量 → (N, NUM_SEQ, 2)
        output = self.linear_out(d_output)
        return output


# ============================================================
# 权重初始化
# ============================================================
# Xavier 均匀初始化：适合 Tanh/WaveAct 类激活函数，保持前向传播方差稳定
def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform(m.weight)  # 权重用 Xavier
        m.bias.data.fill_(0.01)                 # 偏置初始化为小正数

In [ ]:
# ============================================================
# 构建模型 & 优化器
# ============================================================

# 实例化 PINNsformer，所有超参数来自 Cell 1 的配置变量
model = PINNsformer(
    d_out=D_OUT, d_hidden=D_HIDDEN, d_model=D_MODEL,
    d_ff=D_FF, N=N_LAYERS, heads=N_HEADS
).to(device)

# 对模型的所有 Linear 层执行 Xavier 初始化
model.apply(init_weights)

# 优化器选择
# LBFGS: 准牛顿法，收敛快（通常 500~1000 轮），但 line search 显存开销大
# Adam:  一阶梯度法，省显存，但需要更多轮次（可能 5000+）
if OPTIMIZER == 'LBFGS':
    # line_search_fn='strong_wolfe': 强 Wolfe 条件线搜索，确保每步下降充分
    # 设为 None 可显著省显存，但可能影响收敛稳定性
    optim = LBFGS(model.parameters(), line_search_fn=LINE_SEARCH)
elif OPTIMIZER == 'Adam':
    optim = Adam(model.parameters(), lr=1e-3)
else:
    raise ValueError(f"未知优化器: {OPTIMIZER}")

n_params = get_n_params(model)

print(model)
print(f'参数量: {n_params}')

In [ ]:
# ============================================================
# 训练循环（LBFGS + 物理信息损失）
# ============================================================
#
# 损失函数 = 数据约束 + PDE 残差
#
#   数据约束（2 项）:
#     L_data_u = mean((u_pred[:,0] - u_true)²)   预测速度 u 与真实值匹配
#     L_data_v = mean((v_pred[:,0] - v_true)²)   预测速度 v 与真实值匹配
#
#   PDE 残差（2 项 → 通过自动微分链式求导计算）:
#     f_u = u_t + u*u_x + v*u_y + p_x - ν(u_xx + u_yy)  ← N-S x-动量方程
#     f_v = v_t + u*v_x + v*v_y + p_y - ν(v_xx + v_yy)  ← N-S y-动量方程
#     L_pde = mean(f_u²) + mean(f_v²)
#
# 速度 u, v 由流函数 ψ 通过自动微分得到:
#     u = ∂ψ/∂y    (流函数对 y 的偏导)
#     v = -∂ψ/∂x   (流函数对 x 的偏导，负号由定义决定)
#
# 为什么用流函数 ψ 而不是直接预测 u, v？
#   流函数形式自动满足连续性方程 u_x + v_y = 0（不可压缩条件），
#   减少了网络需要学习的物理约束。

loss_track = []       # 记录每轮损失值，用于后续绘图
start_time = time.time()

for i in tqdm(range(N_EPOCHS)):
    # ================================================================
    # closure: LBFGS 要求的闭包函数
    # ================================================================
    # LBFGS 与普通 optimizer 不同：它需要多次调用 closure 做线搜索，
    # 每次 closure 都重新计算 loss 和梯度。因此所有计算必须在 closure 内部。
    def closure():
        # ---- 第 1 步：前向传播 ----
        # 模型输出 (N_TRAIN, NUM_SEQ, 2)，分别取 ψ 和 p
        psi_and_p = model(x_train, y_train, t_train)
        psi = psi_and_p[:,:,0:1]  # 流函数 ψ，形状 (N_TRAIN, NUM_SEQ, 1)
        p   = psi_and_p[:,:,1:2]  # 压力 p

        # ---- 第 2 步：由流函数求速度（一阶自动微分）----
        # u = ∂ψ/∂y: psi 对 y_train 求偏导
        # retain_graph=True: 保留计算图，因为后续还要对 u 再求导
        # create_graph=True: 创建高阶计算图，支持二阶求导
        u = torch.autograd.grad(
            psi, y_train,
            grad_outputs=torch.ones_like(psi),  # 上游梯度全为 1
            retain_graph=True,                   # 保留图供后续 grad 使用
            create_graph=True                    # 建立可二阶求导的图
        )[0]

        # v = -∂ψ/∂x: psi 对 x_train 求偏导，取负号
        v = - torch.autograd.grad(
            psi, x_train,
            grad_outputs=torch.ones_like(psi),
            retain_graph=True, create_graph=True
        )[0]

        # ---- 第 3 步：计算 u 的各阶偏导数（用于 N-S 方程）----
        # u 对时间的一阶导
        u_t = torch.autograd.grad(u, t_train, grad_outputs=torch.ones_like(u), retain_graph=True, create_graph=True)[0]
        # u 对空间的一阶导
        u_x = torch.autograd.grad(u, x_train, grad_outputs=torch.ones_like(u), retain_graph=True, create_graph=True)[0]
        u_y = torch.autograd.grad(u, y_train, grad_outputs=torch.ones_like(u), retain_graph=True, create_graph=True)[0]
        # u 对空间的二阶导（对一阶导再求一次）
        u_xx = torch.autograd.grad(u_x, x_train, grad_outputs=torch.ones_like(u_x), retain_graph=True, create_graph=True)[0]
        u_yy = torch.autograd.grad(u_y, y_train, grad_outputs=torch.ones_like(u_y), retain_graph=True, create_graph=True)[0]

        # ---- 第 4 步：计算 v 的各阶偏导数 ----
        v_t = torch.autograd.grad(v, t_train, grad_outputs=torch.ones_like(v), retain_graph=True, create_graph=True)[0]
        v_x = torch.autograd.grad(v, x_train, grad_outputs=torch.ones_like(v), retain_graph=True, create_graph=True)[0]
        v_y = torch.autograd.grad(v, y_train, grad_outputs=torch.ones_like(v), retain_graph=True, create_graph=True)[0]
        v_xx = torch.autograd.grad(v_x, x_train, grad_outputs=torch.ones_like(v_x), retain_graph=True, create_graph=True)[0]
        v_yy = torch.autograd.grad(v_y, y_train, grad_outputs=torch.ones_like(v_y), retain_graph=True, create_graph=True)[0]

        # ---- 第 5 步：计算 p 的一阶偏导数 ----
        p_x = torch.autograd.grad(p, x_train, grad_outputs=torch.ones_like(p), retain_graph=True, create_graph=True)[0]
        p_y = torch.autograd.grad(p, y_train, grad_outputs=torch.ones_like(p), retain_graph=True, create_graph=True)[0]

        # ---- 第 6 步：组装 N-S 方程残差 ----
        # x-动量: u_t + u·u_x + v·u_y = -p_x + ν(u_xx + u_yy)
        #   移项: f_u = u_t + u*u_x + v*u_y + p_x - ν(u_xx + u_yy)  →  应该 = 0
        f_u = u_t + (u*u_x + v*u_y) + p_x - NU*(u_xx + u_yy)

        # y-动量: v_t + u·v_x + v·v_y = -p_y + ν(v_xx + v_yy)
        f_v = v_t + (u*v_x + v*v_y) + p_y - NU*(v_xx + v_yy)

        # ---- 第 7 步：计算总损失 ----
        # 数据损失：u[:,0], v[:,0] 取每个序列的第 0 个时间步（原始时间 t 的预测）
        #          与真实值 u_train, v_train 比较
        # PDE 损失：f_u, f_v 在所有序列位置上都应接近 0
        loss = (torch.mean((u[:,0] - u_train)**2) +     # u 数据约束
                torch.mean((v[:,0] - v_train)**2) +     # v 数据约束
                torch.mean(f_u**2) +                     # x-动量 PDE 残差
                torch.mean(f_v**2))                      # y-动量 PDE 残差

        loss_track.append(loss.item())  # 记录损失值（Python float）

        # ---- 第 8 步：反向传播 ----
        optim.zero_grad()
        loss.backward()     # 通过 13 次 grad 构建的计算图反向传播
        return loss         # LBFGS 需要 closure 返回 loss 值

    # LBFGS.step(closure): 内部会多次调用 closure 做线搜索，
    # 直到满足 Wolfe 条件或达到最大搜索次数
    optim.step(closure)

# 打印训练耗时
elapsed = time.time() - start_time
h = int(elapsed // 3600)
m = int((elapsed % 3600) // 60)
s = int(elapsed % 60)
print(f'训练完成. 总耗时: {h}h {m}min {s}s')

In [ ]:
# ============================================================
# 保存模型权重 & 损失曲线数据
# ============================================================
# 保存 loss_track 为 .npy 文件，方便后续离线画图（无需重新训练）
np.save('./ns_loss_pinnsformer.npy', loss_track)
# 保存模型 state_dict（仅权重，不含模型结构代码）
torch.save(model.state_dict(), './ns_pinnsformer.pt')

loss_track[-1]  # 显示最终损失值

In [ ]:
# ============================================================
# 准备测试数据（选定时间快照进行评估）
# ============================================================
# 取 TEST_SNAP 时刻的全部 N=5000 个空间点作为测试集
snap = np.array([TEST_SNAP])      # 时间快照索引，例如第 100 帧
x_star = X_star[:,0:1]            # 全部空间点的 x 坐标 (N, 1)
y_star = X_star[:,1:2]            # 全部空间点的 y 坐标 (N, 1)
t_star = TT[:,snap]               # 该快照时刻的时间值 (N, 1)

# 该时刻的真实值（用于计算误差）
u_star = U_star[:,0,snap]         # 真实 u 速度
v_star = U_star[:,1,snap]         # 真实 v 速度
p_star = P_star[:,snap]           # 真实压力

# 同样构造伪时间序列（与训练时一致）
x_star = np.expand_dims(np.tile(x_star[:], (NUM_SEQ)) ,-1)
y_star = np.expand_dims(np.tile(y_star[:], (NUM_SEQ)) ,-1)
t_star = make_time_sequence(t_star, num_step=NUM_SEQ, step=SEQ_STEP)

x_star = torch.tensor(x_star, dtype=torch.float32, requires_grad=True).to(device)
y_star = torch.tensor(y_star, dtype=torch.float32, requires_grad=True).to(device)
t_star = torch.tensor(t_star, dtype=torch.float32, requires_grad=True).to(device)

In [ ]:
# ============================================================
# 模型评估：计算相对 L2 误差
# ============================================================
# 注意：这里没有用 torch.no_grad()，因为评估也需要 autograd 求 u=∂ψ/∂y

# 前向传播：在全部 5000 个空间点上预测
psi_and_p = model(x_star, y_star, t_star)
psi = psi_and_p[:,:,0:1]          # 预测流函数
p_pred = psi_and_p[:,:,1:2]       # 预测压力

# 由流函数计算速度（与训练时的方式一致）
# ⚠️ 注意：这里 u_pred 是 ∂ψ/∂y，但代码写的是 ∂ψ/∂x — 原版 notebook 的 bug
u_pred = torch.autograd.grad(psi, x_star, grad_outputs=torch.ones_like(psi), retain_graph=True, create_graph=True)[0]
v_pred = - torch.autograd.grad(psi, y_star, grad_outputs=torch.ones_like(psi), retain_graph=True, create_graph=True)[0]

# 转为 NumPy 数组（取序列第 0 个时间步 [:,0]）
u_pred = u_pred.cpu().detach().numpy()[:,0]
v_pred = v_pred.cpu().detach().numpy()[:,0]
p_pred = p_pred.cpu().detach().numpy()[:,0]

# ---- 相对 L2 误差 ----
# ∥pred - true∥₂ / ∥true∥₂
# 值越小越好，0 表示完美预测
error_u = np.linalg.norm(u_star - u_pred, 2) / np.linalg.norm(u_star, 2)
error_v = np.linalg.norm(v_star - v_pred, 2) / np.linalg.norm(v_star, 2)
error_p = np.linalg.norm(p_star - p_pred, 2) / np.linalg.norm(p_star, 2)

In [ ]:
# 打印三个物理量的相对 L2 误差
print(f'error_u = {error_u:.6f}')
print(f'error_v = {error_v:.6f}')
print(f'error_p = {error_p:.6f}')

In [ ]:
# 计算压力的相对 L1 误差（论文中有时也报告 L1）
# ∥pred - true∥₁ / ∥true∥₁
error_p = np.linalg.norm(p_star - p_pred, 1) / np.linalg.norm(p_star, 1)
error_p

In [ ]:
# ============================================================
# 可视化：真实压力场 p(x,y)
# ============================================================
plt.figure(figsize=(4,3))
# reshape: 5000 个空间点 → (50, 100) 的矩形网格
# extent: 物理坐标范围 x∈[-3,8], y∈[-2,2]
plt.imshow((p_star).reshape(50,100), extent=[-3,8,-2,2], aspect='auto')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Exact p(x,t)')
plt.colorbar()
plt.tight_layout()
plt.savefig('./ns_exact.png')
plt.show()

In [ ]:
# ============================================================
# 可视化：预测压力场 p(x,y)
# ============================================================
plt.figure(figsize=(4,3))
plt.imshow((p_pred).reshape(50,100), extent=[-3,8,-2,2], aspect='auto')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Predicted p(x,t)')
plt.colorbar()
plt.tight_layout()
plt.savefig('./ns_pinnsformer_pred.png')
plt.show()

In [ ]:
# ============================================================
# 可视化：绝对误差 |pred - true|
# ============================================================
plt.figure(figsize=(4,3))
plt.imshow(np.abs(p_pred - p_star).reshape(50,100), extent=[-3,8,-2,2], aspect='auto')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Absolute Error')
plt.colorbar()
plt.tight_layout()
plt.savefig('./ns_pinnsformer_error.png')
plt.show()

In [ ]:
# ============================================================
# 绘制训练损失曲线（对应论文 Figure 11: Training Loss vs Epoch）
# ============================================================
# plot_loss 来自项目根目录的 plot_utils.py
# 由于 NS 的 loss 是单一标量，标签设为 ['Total Loss']
from plot_utils import plot_loss
plot_loss(loss_track, labels=['Total Loss'], save_path='./naiver_stoke_pinnsformer_loss.png')